# 🚗 Vehicle Testing Dashboard — CAN Bus Data Analysis

**Purpose:** Interactive dashboard to analyze simulated CAN bus data from vehicle track/lab tests, detect early warning signs of hardware failure (cooling system, alternator), compare component versions (e.g. Suspension v1 vs v2), and track calibration status of test equipment.

**How to use this notebook**
1. Run all cells top to bottom (`Kernel > Restart & Run All`).
2. Use the filters in **Section A** to select which `Test_ID`(s) and `Component_Version` to analyze.
3. Explore the interactive plots in **Section B**.
4. Check upcoming/overdue calibration in **Section C**.
5. Review the automated fault report in **Section D**.
6. Exported PNG snapshots are saved to `/reports` at the end of the notebook, ready to drop into a test report or PPT.

> This notebook simulates data acquisition and does not connect to a real CAN bus / DAQ system. Replace `generate_can_bus_data()` with your real data ingestion pipeline (e.g. MDF4/ASC log parser) when moving to production.

In [1]:
# %pip install plotly ipywidgets pandas numpy kaleido -q

## Imports & Setup

We use:
- `pandas` / `numpy` for data handling and simulation
- `sqlite3` for a lightweight local database (stand-in for a real test-data acquisition DB)
- `plotly` for interactive, web-native charts (no `%matplotlib widget` needed — Plotly is interactive by default in Jupyter via its own renderer)
- `ipywidgets` for the interactive filter panel

In [2]:
import numpy as np
import pandas as pd
import sqlite3
import os
from datetime import datetime, timedelta

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

import warnings
warnings.filterwarnings('ignore')

# Plotly renders natively/interactively inside Jupyter — this is our equivalent
# of enabling '%matplotlib widget' for the matplotlib ecosystem.
pio.renderers.default = "notebook"

np.random.seed(42)
print("Environment ready.")


Environment ready.


## Simulated CAN Bus Data Acquisition

`generate_can_bus_data()` simulates a single track test session at 1 Hz, producing the signals a real DAQ/CAN logger would capture: speed, RPM, coolant temperature, battery voltage, brake pressure, 3-axis acceleration, and gear.

Two failure modes are **intentionally injected** so the dashboard has something real to detect:
- A **coolant temperature spike** above 105 °C (simulated cooling system issue).
- A **battery voltage drop** below 11.5 V (simulated alternator/charging issue).

Set `inject_anomalies=False` to generate a "clean" baseline run (useful for comparing v1 vs v2 components).

In [3]:
def generate_can_bus_data(test_id, vehicle_vin, start_time=None, duration_seconds=600,
                           sample_rate_hz=1, inject_anomalies=True, seed_offset=0):
    """
    Simulate a single vehicle test run's CAN bus time series.

    Parameters
    ----------
    test_id : str            Identifier of the test run (e.g. 'TEST_001')
    vehicle_vin : str        VIN of the test vehicle
    start_time : datetime    Start timestamp of the run (defaults to now)
    duration_seconds : int   Length of the test run in seconds
    sample_rate_hz : float   Sampling rate
    inject_anomalies : bool  Whether to inject a coolant/voltage anomaly
    seed_offset : int        Offsets the RNG so different runs aren't identical

    Returns
    -------
    pd.DataFrame with columns:
        Timestamp, Test_ID, Vehicle_Speed_kmh, Engine_RPM, Coolant_Temp_C,
        Battery_Voltage, Brake_Pressure_Bar, Acceleration_X, Acceleration_Y,
        Acceleration_Z, Gear
    """
    rng = np.random.default_rng(42 + seed_offset)
    if start_time is None:
        start_time = datetime.now() - timedelta(seconds=duration_seconds)

    n = int(duration_seconds * sample_rate_hz)
    t = np.arange(n)
    timestamps = [start_time + timedelta(seconds=i / sample_rate_hz) for i in t]

    # --- Speed profile: accel -> cruise -> brake -> repeat (simulates track laps) ---
    cycle = 120  # seconds per accel/cruise/brake cycle
    phase = (t % cycle) / cycle
    speed = np.where(
        phase < 0.35, 140 * (phase / 0.35),                       # acceleration
        np.where(phase < 0.7, 140 + 5 * np.sin(phase * 20),       # cruise w/ small variation
                 140 * (1 - (phase - 0.7) / 0.3))                 # braking
    )
    speed = np.clip(speed + rng.normal(0, 1.5, n), 0, None)

    # --- Gear derived from speed ---
    gear_bins = [0, 20, 40, 65, 95, 125, 300]
    gear = np.digitize(speed, gear_bins)

    # --- Engine RPM correlated with speed & gear ---
    rpm = 900 + (speed / (gear + 0.3)) * 55 + rng.normal(0, 80, n)
    rpm = np.clip(rpm, 800, 7000)

    # --- Coolant temperature: warms up, plateaus, small noise ---
    warmup = np.clip(t / 180, 0, 1)  # reaches steady state after ~3 min
    coolant_temp = 25 + warmup * (90 - 25) + rng.normal(0, 0.8, n)
    coolant_temp += 0.01 * np.clip(rpm - 4000, 0, None)  # extra heat under high load

    # --- Battery voltage: alternator charging while running ---
    battery_voltage = 13.8 + rng.normal(0, 0.15, n) - 0.0003 * np.clip(rpm - 5000, 0, None)
    battery_voltage = np.clip(battery_voltage, 11.8, 14.6)

    # --- Brake pressure: spikes during the braking phase of each cycle ---
    braking_mask = phase >= 0.7
    brake_pressure = np.where(braking_mask, rng.uniform(20, 90, n), rng.uniform(0, 2, n))

    # --- 3-axis acceleration (g), influenced by braking / cornering / speed changes ---
    accel_x = np.gradient(speed) * 0.15 + rng.normal(0, 0.05, n)          # longitudinal
    accel_y = np.sin(t / 15) * 0.25 * (speed / 140) + rng.normal(0, 0.04, n)  # lateral (cornering)
    accel_z = 1.0 + rng.normal(0, 0.03, n) + (braking_mask * 0.08)        # vertical (~1g + road/braking noise)

    df = pd.DataFrame({
        "Timestamp": timestamps,
        "Test_ID": test_id,
        "Vehicle_Speed_kmh": speed.round(1),
        "Engine_RPM": rpm.round(0),
        "Coolant_Temp_C": coolant_temp.round(1),
        "Battery_Voltage": battery_voltage.round(2),
        "Brake_Pressure_Bar": brake_pressure.round(1),
        "Acceleration_X": accel_x.round(3),
        "Acceleration_Y": accel_y.round(3),
        "Acceleration_Z": accel_z.round(3),
        "Gear": gear.astype(int),
    })

    # --- Inject anomalies: a short coolant spike + a short voltage drop ---
    if inject_anomalies and n > 200:
        # Coolant temperature spike (simulated cooling system fault)
        spike_start = rng.integers(150, n - 100)
        spike_len = rng.integers(8, 20)
        spike_end = min(spike_start + spike_len, n)
        df.loc[spike_start:spike_end, "Coolant_Temp_C"] = rng.uniform(106, 114, spike_end - spike_start + 1)

        # Battery voltage drop (simulated alternator fault)
        drop_start = rng.integers(50, n - 150)
        drop_len = rng.integers(6, 15)
        drop_end = min(drop_start + drop_len, n)
        df.loc[drop_start:drop_end, "Battery_Voltage"] = rng.uniform(10.8, 11.4, drop_end - drop_start + 1)

    return df

# Quick sanity check
_sample = generate_can_bus_data("TEST_SAMPLE", "VIN0000TEST", duration_seconds=60)
_sample.head()


,Timestamp,Test_ID,Vehicle_Speed_kmh,Engine_RPM,Coolant_Temp_C,Battery_Voltage,Brake_Pressure_Bar,Acceleration_X,Acceleration_Y,Acceleration_Z,Gear
0,2026-08-13 14:01:41.009539,TEST_SAMPLE,0.5,800.0,24.2,14.00,0.5,0.251,-0.011,1.019,1
1,2026-08-13 14:01:42.009539,TEST_SAMPLE,1.8,948.0,25.5,13.83,0.3,0.562,0.008,0.990,1
2,2026-08-13 14:01:43.009539,TEST_SAMPLE,7.8,1243.0,25.9,13.74,1.0,0.735,0.045,1.032,1
3,2026-08-13 14:01:44.009539,TEST_SAMPLE,11.4,1430.0,27.2,13.97,0.8,0.210,0.057,0.966,1
4,2026-08-13 14:01:45.009539,TEST_SAMPLE,10.4,1397.0,27.1,13.86,0.5,0.205,0.002,1.000,1


## Data Acquisition Database (SQLite)

In a real validation lab, telemetry would land in an acquisition database keyed by test run and vehicle metadata. Here we build a local SQLite database with two tables:

- **`test_runs`** — the time series signals for every simulated test.
- **`vehicle_metadata`** — one row per test, with `Test_ID`, `Vehicle_VIN`, `Component_Version` (v1 vs v2, so we can compare hardware revisions), and `Calibration_Due_Date` for the test equipment/vehicle instrumentation.

Three test runs are generated: two on **Suspension v1** (one with injected anomalies, one overdue on calibration) and one clean run on **Suspension v2**, so the version-comparison filter has something meaningful to show.

In [4]:
def build_database(db_path="vehicle_testing.db"):
    """Create/replace the SQLite database with simulated test_runs and vehicle_metadata tables."""
    try:
        conn = sqlite3.connect(db_path)

        test_configs = [
            {
                "Test_ID": "TEST_001",
                "Vehicle_VIN": "VIN0001XXXXXXXXX",
                "Component_Version": "Suspension v1",
                "Calibration_Due_Date": (datetime.now() + timedelta(days=6)).strftime("%Y-%m-%d"),
                "inject_anomalies": True,
                "seed_offset": 1,
            },
            {
                "Test_ID": "TEST_002",
                "Vehicle_VIN": "VIN0002XXXXXXXXX",
                "Component_Version": "Suspension v2",
                "Calibration_Due_Date": (datetime.now() + timedelta(days=52)).strftime("%Y-%m-%d"),
                "inject_anomalies": False,
                "seed_offset": 2,
            },
            {
                "Test_ID": "TEST_003",
                "Vehicle_VIN": "VIN0003XXXXXXXXX",
                "Component_Version": "Suspension v1",
                "Calibration_Due_Date": (datetime.now() - timedelta(days=3)).strftime("%Y-%m-%d"),  # overdue
                "inject_anomalies": True,
                "seed_offset": 3,
            },
        ]

        all_runs = []
        base_start = datetime.now() - timedelta(hours=2)
        for cfg in test_configs:
            df = generate_can_bus_data(
                cfg["Test_ID"], cfg["Vehicle_VIN"],
                start_time=base_start,
                duration_seconds=600,
                inject_anomalies=cfg["inject_anomalies"],
                seed_offset=cfg["seed_offset"],
            )
            all_runs.append(df)

        full_runs_df = pd.concat(all_runs, ignore_index=True)
        full_runs_df.to_sql("test_runs", conn, if_exists="replace", index=False)

        metadata_df = pd.DataFrame([
            {k: v for k, v in cfg.items() if k not in ("inject_anomalies", "seed_offset")}
            for cfg in test_configs
        ])
        metadata_df.to_sql("vehicle_metadata", conn, if_exists="replace", index=False)

        conn.commit()
        print(f"Database '{db_path}' built successfully "
              f"({len(full_runs_df)} rows in test_runs, {len(metadata_df)} rows in vehicle_metadata).")
        return conn
    except Exception as e:
        print(f"[ERROR] Failed to build database: {e}")
        return None


conn = build_database()


Database 'vehicle_testing.db' built successfully (1800 rows in test_runs, 3 rows in vehicle_metadata).


### Robust data loading

Loading is wrapped in `try/except` so a missing/corrupt database doesn't crash the whole dashboard — it instead falls back to empty DataFrames and prints a clear error for the engineer.

In [5]:
def load_data(conn):
    try:
        test_runs = pd.read_sql("SELECT * FROM test_runs", conn, parse_dates=["Timestamp"])
        metadata = pd.read_sql("SELECT * FROM vehicle_metadata", conn, parse_dates=["Calibration_Due_Date"])
        return test_runs, metadata
    except Exception as e:
        print(f"[ERROR] Failed to load data from database: {e}")
        return pd.DataFrame(), pd.DataFrame()


test_runs_df, metadata_df = load_data(conn)
print(f"Loaded {len(test_runs_df)} telemetry rows across {test_runs_df['Test_ID'].nunique()} test runs.")
display(metadata_df)


Loaded 1800 telemetry rows across 3 test runs.


,Test_ID,Vehicle_VIN,Component_Version,Calibration_Due_Date
0,TEST_001,VIN0001XXXXXXXXX,Suspension v1,2026-08-19
1,TEST_002,VIN0002XXXXXXXXX,Suspension v2,2026-10-04
2,TEST_003,VIN0003XXXXXXXXX,Suspension v1,2026-08-10


## Automated Fault Detection

`detect_anomalies()` flags every sample where:
- `Coolant_Temp_C > 105°C` → possible **cooling system** issue, or
- `Battery_Voltage < 11.5V` → possible **alternator/charging** issue.

`generate_report()` turns those flags into a plain-language Markdown summary for a product/test engineer, in the format:

> *"Potential issue detected in Test X at timestamp Y. Recommend checking cooling system / alternator."*

In [6]:
COOLANT_THRESHOLD_C = 105.0
VOLTAGE_THRESHOLD_V = 11.5


def detect_anomalies(df):
    """Flag rows exceeding safe coolant temp / battery voltage thresholds."""
    flagged = df.copy()
    flagged["Anomaly_Flag"] = "Normal"
    flagged.loc[flagged["Coolant_Temp_C"] > COOLANT_THRESHOLD_C, "Anomaly_Flag"] = "High Coolant Temp"
    flagged.loc[flagged["Battery_Voltage"] < VOLTAGE_THRESHOLD_V, "Anomaly_Flag"] = "Low Battery Voltage"
    issues = flagged[flagged["Anomaly_Flag"] != "Normal"].copy()
    return flagged, issues


def generate_report(df, max_events=10):
    """Render a Markdown report advising the engineer of detected issues."""
    _, issues = detect_anomalies(df)

    if issues.empty:
        display(Markdown("### ✅ No anomalies detected in the selected test run(s)."))
        return

    # Collapse consecutive flagged samples into discrete "events" per Test_ID/Anomaly_Flag
    issues = issues.sort_values(["Test_ID", "Timestamp"])
    lines = ["### ⚠️ Automated Fault Detection Report", ""]
    event_count = 0

    for (test_id, flag), grp in issues.groupby(["Test_ID", "Anomaly_Flag"]):
        component = "cooling system" if flag == "High Coolant Temp" else "alternator"
        first_ts = grp["Timestamp"].iloc[0]
        peak = grp["Coolant_Temp_C"].max() if flag == "High Coolant Temp" else grp["Battery_Voltage"].min()
        unit = "°C" if flag == "High Coolant Temp" else "V"
        lines.append(
            f"- **Potential issue detected in {test_id} at {first_ts}.** "
            f"{flag} (peak value: {peak:.1f}{unit}). "
            f"Recommend checking {component}."
        )
        event_count += 1
        if event_count >= max_events:
            lines.append(f"- ...and {len(issues) - max_events} more flagged samples not shown.")
            break

    display(Markdown("\n".join(lines)))


## Dashboard — Section A: Filters

Filter by:
- **Test_ID** (multi-select) — choose one or more runs to overlay/compare
- **Component_Version** — restrict to `Suspension v1`, `Suspension v2`, or `All`
- **Date range** — restrict the time window shown within the selected run(s)

In [7]:
test_id_options = sorted(metadata_df["Test_ID"].unique().tolist())
version_options = ["All"] + sorted(metadata_df["Component_Version"].unique().tolist())

test_id_selector = widgets.SelectMultiple(
    options=test_id_options,
    value=tuple(test_id_options[:1]),
    description="Test_ID:",
    style={"description_width": "initial"},
)

version_toggle = widgets.ToggleButtons(
    options=version_options,
    description="Version:",
    style={"description_width": "initial"},
)

min_ts, max_ts = test_runs_df["Timestamp"].min(), test_runs_df["Timestamp"].max()
date_options = pd.date_range(min_ts, max_ts, periods=50)
date_range_slider = widgets.SelectionRangeSlider(
    options=[(d.strftime("%H:%M:%S"), d) for d in date_options],
    index=(0, len(date_options) - 1),
    description="Time window:",
    layout=widgets.Layout(width="500px"),
    style={"description_width": "initial"},
)

filters_box = widgets.VBox([
    widgets.HTML("<h3>Section A — Filters</h3>"),
    test_id_selector,
    version_toggle,
    date_range_slider,
])
filters_box


## Dashboard — Section B: Dynamic Signal Plots

- **Speed vs RPM** (dual Y-axis line chart), with coolant temp / battery voltage plotted below and anomalies marked in red.
- **3D scatter of X/Y/Z acceleration**, colored by speed, to visualize chassis dynamic behavior (braking, cornering, road input) at a glance.

In [8]:
def plot_speed_rpm(df):
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(x=df["Timestamp"], y=df["Vehicle_Speed_kmh"],
                              name="Speed (km/h)", line=dict(color="royalblue")), secondary_y=False)
    fig.add_trace(go.Scatter(x=df["Timestamp"], y=df["Engine_RPM"],
                              name="Engine RPM", line=dict(color="darkorange")), secondary_y=True)
    fig.update_yaxes(title_text="Speed (km/h)", secondary_y=False)
    fig.update_yaxes(title_text="Engine RPM", secondary_y=True)
    fig.update_layout(title="Vehicle Speed vs Engine RPM", height=350,
                       legend=dict(orientation="h", y=1.15))
    return fig


def plot_temp_voltage(df):
    flagged, issues = detect_anomalies(df)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                         subplot_titles=("Coolant Temperature (°C)", "Battery Voltage (V)"))

    fig.add_trace(go.Scatter(x=df["Timestamp"], y=df["Coolant_Temp_C"],
                              mode="lines", name="Coolant Temp", line=dict(color="seagreen")), row=1, col=1)
    temp_issues = issues[issues["Anomaly_Flag"] == "High Coolant Temp"]
    if not temp_issues.empty:
        fig.add_trace(go.Scatter(x=temp_issues["Timestamp"], y=temp_issues["Coolant_Temp_C"],
                                  mode="markers", marker=dict(color="red", size=7),
                                  name="Temp Anomaly"), row=1, col=1)
    fig.add_hline(y=COOLANT_THRESHOLD_C, line_dash="dot", line_color="red", row=1, col=1)

    fig.add_trace(go.Scatter(x=df["Timestamp"], y=df["Battery_Voltage"],
                              mode="lines", name="Battery Voltage", line=dict(color="purple")), row=2, col=1)
    volt_issues = issues[issues["Anomaly_Flag"] == "Low Battery Voltage"]
    if not volt_issues.empty:
        fig.add_trace(go.Scatter(x=volt_issues["Timestamp"], y=volt_issues["Battery_Voltage"],
                                  mode="markers", marker=dict(color="red", size=7),
                                  name="Voltage Anomaly"), row=2, col=1)
    fig.add_hline(y=VOLTAGE_THRESHOLD_V, line_dash="dot", line_color="red", row=2, col=1)

    fig.update_layout(height=550, showlegend=True, title_text="Cooling System & Charging System — Anomaly View")
    return fig


def plot_acceleration_3d(df):
    fig = px.scatter_3d(
        df, x="Acceleration_X", y="Acceleration_Y", z="Acceleration_Z",
        color="Vehicle_Speed_kmh", color_continuous_scale="Viridis",
        title="Chassis Dynamic Behavior — 3-Axis Acceleration (colored by speed)",
        labels={"Acceleration_X": "Longitudinal (g)", "Acceleration_Y": "Lateral (g)", "Acceleration_Z": "Vertical (g)"},
    )
    fig.update_traces(marker=dict(size=3, opacity=0.7))
    fig.update_layout(height=550)
    return fig


## Dashboard — Section C: Calibration / Maintenance Status

Bar chart of days remaining until each test vehicle/equipment's calibration is due, color-coded:
- 🟢 **OK** — more than 30 days remaining
- 🟠 **Due Soon** — within 30 days
- 🔴 **Overdue** — past due date, do not use for certified testing

In [9]:
def plot_calibration_status(metadata):
    meta = metadata.copy()
    meta["Calibration_Due_Date"] = pd.to_datetime(meta["Calibration_Due_Date"])
    meta["Days_Remaining"] = (meta["Calibration_Due_Date"] - pd.Timestamp.now()).dt.days

    def status(days):
        if days < 0:
            return "Overdue"
        elif days <= 30:
            return "Due Soon"
        return "OK"

    meta["Status"] = meta["Days_Remaining"].apply(status)
    color_map = {"Overdue": "red", "Due Soon": "orange", "OK": "green"}

    fig = px.bar(
        meta, x="Vehicle_VIN", y="Days_Remaining", color="Status",
        color_discrete_map=color_map,
        title="Calibration Status by Vehicle / Test Equipment",
        hover_data=["Test_ID", "Component_Version", "Calibration_Due_Date"],
    )
    fig.add_hline(y=0, line_dash="dash", line_color="black")
    fig.update_layout(height=400)
    return fig


## Assembling the Interactive Dashboard

Ties Sections A–C together: changing any filter re-renders every chart and the fault report automatically.

In [10]:
out_speed = widgets.Output()
out_temp = widgets.Output()
out_3d = widgets.Output()
out_calib = widgets.Output()
out_report = widgets.Output()


def get_filtered_data():
    selected_ids = list(test_id_selector.value)

    if version_toggle.value != "All":
        valid_ids = metadata_df.loc[metadata_df["Component_Version"] == version_toggle.value, "Test_ID"].tolist()
        selected_ids = [t for t in selected_ids if t in valid_ids]

    if not selected_ids:
        selected_ids = test_id_options

    df = test_runs_df[test_runs_df["Test_ID"].isin(selected_ids)]

    start_dt, end_dt = date_range_slider.value
    df = df[(df["Timestamp"] >= start_dt) & (df["Timestamp"] <= end_dt)]
    return df


def update_dashboard(change=None):
    try:
        filtered = get_filtered_data()

        with out_speed:
            clear_output(wait=True)
            if filtered.empty:
                print("No data for the current filter selection.")
            else:
                plot_speed_rpm(filtered).show()

        with out_temp:
            clear_output(wait=True)
            if not filtered.empty:
                plot_temp_voltage(filtered).show()

        with out_3d:
            clear_output(wait=True)
            if filtered.empty:
                print("No data for the current filter selection.")
            else:
                plot_acceleration_3d(filtered).show()

        with out_calib:
            clear_output(wait=True)
            plot_calibration_status(metadata_df).show()

        with out_report:
            clear_output(wait=True)
            if filtered.empty:
                display(Markdown("_No data available for the fault report._"))
            else:
                generate_report(filtered)

    except Exception as e:
        print(f"[ERROR] Dashboard update failed: {e}")


test_id_selector.observe(update_dashboard, names="value")
version_toggle.observe(update_dashboard, names="value")
date_range_slider.observe(update_dashboard, names="value")

tab = widgets.Tab()
tab.children = [
    widgets.VBox([out_speed, out_temp]),
    out_3d,
    out_calib,
    out_report,
]
tab.set_title(0, "📈 Dynamic Signals")
tab.set_title(1, "🎯 Chassis Dynamics (3D)")
tab.set_title(2, "🛠️ Calibration Status")
tab.set_title(3, "📋 Fault Report")

display(filters_box, tab)
update_dashboard()


## Exporting Charts for Reports

Saves static PNG snapshots of the key charts (based on the current filter selection) to a local `/reports` folder, ready to paste into a test report, DVP&R, or PowerPoint. This uses `kaleido` as the Plotly static-image engine and is wrapped in `try/except` in case `kaleido` isn't installed in the current environment.

In [11]:
# %pip install -U kaleido

In [12]:
os.makedirs("reports", exist_ok=True)

try:
    export_df = get_filtered_data()
    if export_df.empty:
        export_df = test_runs_df

    exports = {
        "reports/speed_vs_rpm.png": plot_speed_rpm(export_df),
        "reports/temp_voltage_anomalies.png": plot_temp_voltage(export_df),
        "reports/acceleration_3d.png": plot_acceleration_3d(export_df),
        "reports/calibration_status.png": plot_calibration_status(metadata_df),
    }

    for path, fig in exports.items():
        fig.write_image(path, scale=2)
        print(f"Saved: {path}")

    print("\nAll charts exported to /reports.")

except Exception as e:
    print(f"[WARNING] Could not export PNG charts (is 'kaleido' installed?). "
          f"Run '%pip install -U kaleido' and re-run this cell.\nDetails: {e}")


Saved: reports/speed_vs_rpm.png
Saved: reports/temp_voltage_anomalies.png
Saved: reports/acceleration_3d.png
[WARNING] Could not export PNG charts (is 'kaleido' installed?). Run '%pip install -U kaleido' and re-run this cell.
Details: Type is not JSON serializable: Timestamp


## Summary for the Product/Test Engineer

- Data for 3 simulated test runs (`TEST_001`, `TEST_002`, `TEST_003`) is stored in SQLite (`vehicle_testing.db`), covering **Suspension v1** and **v2** builds.
- Two faults were intentionally injected to validate the detection logic: a **coolant over-temperature event** and a **battery under-voltage event**, both flagged automatically and marked in red on the relevant charts.
- Use the **Version toggle** in Section A to compare v1 vs v2 behavior side by side (select the matching `Test_ID`s).
- **TEST_003**'s calibration is shown as **Overdue** — its data should be treated as informational only until the equipment is recalibrated.
- Exported chart snapshots are in `/reports` for inclusion in formal test documentation.